# Modelado de Machine Learning
## Prediccion de Precio de Alquiler - Ecuador

**PoliSistemas - Proceso de Seleccion Tecnico de Investigacion**

### Variables de entrada
- `provincia` - Provincia del inmueble
- `lugar` - Ciudad/canton (normalizado desde EDA)
- `num_dormitorios` - Numero de dormitorios
- `num_banos` - Numero de banos
- `area` - Area en m2
- `num_garages` - Numero de garajes

### Variable objetivo
- `precio` - Precio mensual de alquiler (USD)


## 0. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modelos candidatos
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
os.makedirs('model', exist_ok=True)
print('Librerias cargadas correctamente.')

## 1. Carga y Preparacion del Dataset

In [ ]:
import re

def limpiar_lugar(lugar_raw: str, provincia: str) -> str:
    """Extrae ciudad principal desde string de direccion completa."""
    if pd.isna(lugar_raw):
        return provincia
    partes = [p.strip() for p in str(lugar_raw).split(',')]
    partes_limpias = []
    for p in partes:
        if not p or p.lower() == 'ecuador':
            continue
        if re.match(r'^[A-Z0-9]{4}\+[A-Z0-9]+', p) or re.match(r'^\d+$', p):
            continue
        partes_limpias.append(p)
    if not partes_limpias:
        return provincia
    if partes_limpias[0].lower().replace(' ', '') == provincia.lower().replace(' ', ''):
        partes_limpias = partes_limpias[1:]
    if not partes_limpias:
        return provincia
    ciudad = re.sub(r'\s+\d{5,6}\b', '', partes_limpias[-1]).strip()
    return ciudad if ciudad else provincia

# Intentar cargar dataset limpio; si no existe, generarlo
try:
    df = pd.read_csv('data/dataset_limpio.csv')
    print('Dataset limpio cargado desde data/dataset_limpio.csv')
except FileNotFoundError:
    print('Generando dataset limpio...')
    df = pd.read_csv('data/real_state_ecuador_dataset.csv')
    df.columns = ['titulo', 'precio', 'provincia', 'lugar_raw', 'num_dormitorios', 'num_banos', 'area', 'num_garages']
    for col in ['num_dormitorios', 'num_banos', 'area', 'num_garages', 'precio']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['num_dormitorios', 'num_banos', 'area', 'num_garages']:
        df[col] = df[col].fillna(df.groupby('provincia')[col].transform('median')).fillna(df[col].median())
    df = df.dropna(subset=['precio'])
    df['lugar'] = df.apply(lambda r: limpiar_lugar(r['lugar_raw'], r['provincia']), axis=1)
    df.to_csv('data/dataset_limpio.csv', index=False)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
FEATURES = ['provincia', 'lugar', 'num_dormitorios', 'num_banos', 'area', 'num_garages']
TARGET = 'precio'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f'Features: {FEATURES}')
print(f'Target: {TARGET}')
print(f'\nX shape: {X.shape}')
print(f'y - rango: ${y.min():.0f} a ${y.max():.0f} | media: ${y.mean():.0f} | mediana: ${y.median():.0f}')
print(f'\nDistribucion de y (skewness = {y.skew():.2f}):')
y.describe()

## 2. Justificacion del Modelo

### Analisis del problema

La variable objetivo `precio` presenta:
- **Sesgo fuerte a la derecha** (skewness > 1): la mayoria de alquileres son economicos, con pocos inmuebles de lujo.
- **Outliers extremos**: rango de $9 a $9,000 USD/mes.
- **Relaciones no lineales**: el precio no crece de forma lineal con el area ni los dormitorios.

### Estrategia adoptada: GBM + Transformacion log1p del target

| Aspecto | Detalle | Solucion |
|---|---|---|
| Distribucion sesgada | Cola derecha pronunciada | log1p(precio) estabiliza la varianza |
| Outliers extremos | Precios de $9 a $9,000 | Log reduce impacto desproporcionado |
| No linealidad | Area x Provincia != lineal | GBM captura interacciones automaticamente |
| Variables mixtas | Categoricas + numericas | Arboles no requieren supuestos |

Se usa `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` que:
1. Transforma el target antes de entrenar
2. Revierte la transformacion automaticamente al predecir
3. Las predicciones finales estan en la escala original (USD)


## 3. Preprocesamiento

In [ ]:
COLS_CATEGORICAS = ['provincia', 'lugar']
COLS_NUMERICAS = ['num_dormitorios', 'num_banos', 'area', 'num_garages']

preprocessor = ColumnTransformer(
    transformers=[
        # OrdinalEncoder con handle_unknown maneja ubicaciones no vistas en produccion
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), COLS_CATEGORICAS),
        # StandardScaler normaliza escala de features numericos
        ('num', StandardScaler(), COLS_NUMERICAS)
    ],
    remainder='drop'
)

print('Preprocesador configurado:')
print(f'  Categoricas (OrdinalEncoder): {COLS_CATEGORICAS}')
print(f'  Numericas   (StandardScaler): {COLS_NUMERICAS}')

## 4. Division Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train)} muestras ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test:  {len(X_test)} muestras ({len(X_test)/len(X)*100:.0f}%)')

# Visualizar distribucion del target en escala log
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribucion de Precio (escala original)', fontweight='bold')
axes[0].set_xlabel('Precio (USD)')
axes[1].hist(np.log1p(y), bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribucion de log1p(Precio)', fontweight='bold')
axes[1].set_xlabel('log1p(Precio)')
plt.suptitle('Efecto de la transformacion logaritmica sobre el target', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/fig_target_transform.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Comparacion de Modelos

In [ ]:
def construir_pipeline_con_log(regresor) -> Pipeline:
    """Construye pipeline con preprocesamiento y transformacion log1p del target."""
    return Pipeline([
        ('pre', preprocessor),
        ('reg', TransformedTargetRegressor(
            regressor=regresor,
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])


def evaluar_modelo(nombre: str, pipeline: Pipeline, X_tr, y_tr, X_te, y_te) -> dict:
    """Entrena, evalua y retorna metricas de un pipeline."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)

    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_r2 = cross_val_score(pipeline, X_tr, y_tr, cv=cv, scoring='r2').mean()

    print(f'{nombre}:')
    print(f'  MAE:   ${mae:.2f}')
    print(f'  RMSE:  ${rmse:.2f}')
    print(f'  R2:    {r2:.4f}')
    print(f'  CV R2: {cv_r2:.4f} (5-fold)')
    print()

    return {'nombre': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv_r2, 'pipeline': pipeline}


# Modelos candidatos - todos con transformacion log1p
modelos_candidatos = [
    ('Ridge',             construir_pipeline_con_log(Ridge(alpha=1.0))),
    ('Decision Tree',     construir_pipeline_con_log(DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE))),
    ('Random Forest',     construir_pipeline_con_log(RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=3, random_state=RANDOM_STATE))),
    ('Gradient Boosting', construir_pipeline_con_log(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, min_samples_leaf=5, random_state=RANDOM_STATE))),
]

print('=== Comparacion de Modelos (con transformacion log1p) ===\n')
resultados = []
for nombre, pipeline in modelos_candidatos:
    res = evaluar_modelo(nombre, pipeline, X_train, y_train, X_test, y_test)
    resultados.append(res)

In [ ]:
# Tabla comparativa
df_resultados = pd.DataFrame([{
    'Modelo': r['nombre'],
    'MAE ($)': f"{r['MAE']:.0f}",
    'RMSE ($)': f"{r['RMSE']:.0f}",
    'R2': f"{r['R2']:.4f}",
    'CV R2 (5-fold)': f"{r['CV_R2']:.4f}"
} for r in resultados])

print('=== Tabla Comparativa de Metricas ===')
print(df_resultados.to_string(index=False))

# Visualizacion comparativa
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
nombres = [r['nombre'] for r in resultados]
colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for ax, metrica, titulo, fmt in [
    (axes[0], 'MAE',  'MAE - menor es mejor (USD)',   '${:.0f}'),
    (axes[1], 'RMSE', 'RMSE - menor es mejor (USD)',  '${:.0f}'),
    (axes[2], 'R2',   'R2 - mayor es mejor',           '{:.3f}'),
]:
    vals = [r[metrica] for r in resultados]
    ax.bar(nombres, vals, color=colores)
    ax.set_title(titulo, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    if metrica == 'R2':
        ax.set_ylim(0, 1)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.02, fmt.format(v), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Comparacion de Modelos de Regresion (target: log1p(precio))', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/fig_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Modelo Final

In [ ]:
# Seleccionar modelo con mejor R2
mejor_resultado = max(resultados, key=lambda r: r['R2'])
modelo_final = mejor_resultado['pipeline']

print(f'Modelo seleccionado: {mejor_resultado["nombre"]}')
print(f'  R2 en test:  {mejor_resultado["R2"]:.4f}')
print(f'  MAE en test: ${mejor_resultado["MAE"]:.2f}')
print(f'  RMSE en test:${mejor_resultado["RMSE"]:.2f}')
print(f'  CV R2:       {mejor_resultado["CV_R2"]:.4f}')

In [ ]:
# Analisis de residuos
y_pred = modelo_final.predict(X_test)
residuos = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Real vs Predicho
lim = max(y_test.max(), y_pred.max()) * 1.05
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=60, edgecolors='none')
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Prediccion perfecta')
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel('Precio Real (USD)'); axes[0].set_ylabel('Precio Predicho (USD)')
axes[0].set_title('Real vs Predicho', fontweight='bold'); axes[0].legend()

# Histograma de residuos
axes[1].hist(residuos, bins=30, color='teal', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Residuo (Real - Predicho)'); axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion de Residuos', fontweight='bold')

# Residuos vs Predicho
axes[2].scatter(y_pred, residuos, alpha=0.5, color='coral', s=60, edgecolors='none')
axes[2].axhline(0, color='red', lw=1.5, linestyle='--')
axes[2].set_xlabel('Precio Predicho (USD)'); axes[2].set_ylabel('Residuo')
axes[2].set_title('Residuos vs Predicho', fontweight='bold')

plt.suptitle(f'Analisis de Residuos - {mejor_resultado["nombre"]}', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/fig_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Residuo medio: ${residuos.mean():.2f}')
print(f'Desv. estandar residuos: ${residuos.std():.2f}')

In [ ]:
# Importancia de variables
# TransformedTargetRegressor envuelve el regresor en .regressor_
try:
    regresor_interno = modelo_final.named_steps['reg'].regressor_
    importancias = regresor_interno.feature_importances_
    feature_names = COLS_CATEGORICAS + COLS_NUMERICAS

    df_imp = pd.DataFrame({'Feature': feature_names, 'Importancia': importancias})
    df_imp = df_imp.sort_values('Importancia', ascending=True)

    plt.figure(figsize=(10, 5))
    bars = plt.barh(df_imp['Feature'], df_imp['Importancia'],
                    color=sns.color_palette('viridis', len(df_imp)))
    plt.title(f'Importancia de Variables - {mejor_resultado["nombre"]}', fontweight='bold', pad=12)
    plt.xlabel('Importancia relativa')
    for bar, val in zip(bars, df_imp['Importancia']):
        plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
    plt.tight_layout()
    plt.savefig('data/fig_importancia_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df_imp.sort_values('Importancia', ascending=False).to_string(index=False))
except Exception as e:
    print(f'Importancia no disponible: {e}')

## 7. Serializacion del Modelo

In [ ]:
MODEL_PATH = 'model/modelo_precio.pkl'
METADATA_PATH = 'model/metadata.json'

# Reentrenar en el dataset completo (train + test) para maxima capacidad
modelo_final.fit(X, y)

# Guardar
joblib.dump(modelo_final, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')

# Verificar carga
modelo_cargado = joblib.load(MODEL_PATH)
y_check = modelo_cargado.predict(X_test.head(5)).round(2)
print(f'Verificacion - Predicciones: {y_check}')
print(f'Valores reales:              {y_test.head(5).values}')

# Metadata
metadata = {
    'modelo': mejor_resultado['nombre'],
    'target_transform': 'log1p / expm1',
    'features': FEATURES,
    'target': TARGET,
    'metricas': {
        'MAE': round(mejor_resultado['MAE'], 2),
        'RMSE': round(mejor_resultado['RMSE'], 2),
        'R2': round(mejor_resultado['R2'], 4),
        'CV_R2_5fold': round(mejor_resultado['CV_R2'], 4)
    },
    'train_samples': len(X_train),
    'test_samples': len(X_test)
}
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'\nMetadata guardada en: {METADATA_PATH}')
print(json.dumps(metadata, indent=2))

## 8. Prueba del Modelo

In [ ]:
# Ejemplo identico al endpoint POST /predict de la API
ejemplo_api = pd.DataFrame([{
    'provincia': 'Pichincha',
    'lugar': 'Quito',
    'num_dormitorios': 3,
    'num_banos': 2,
    'area': 120,
    'num_garages': 1
}])

pred = modelo_cargado.predict(ejemplo_api)[0]
print('Ejemplo POST /predict:')
print('  Input:  Pichincha / Quito / 3 dorm / 2 banos / 120m2 / 1 garage')
print(f'  Output: {{"prediction": {pred:.2f}}}')

In [ ]:
# Multiples escenarios para validacion manual
escenarios = pd.DataFrame([
    {'provincia': 'Pichincha',   'lugar': 'Quito',      'num_dormitorios': 1, 'num_banos': 1, 'area': 45,  'num_garages': 0},
    {'provincia': 'Pichincha',   'lugar': 'Quito',      'num_dormitorios': 2, 'num_banos': 2, 'area': 80,  'num_garages': 1},
    {'provincia': 'Pichincha',   'lugar': 'Quito',      'num_dormitorios': 3, 'num_banos': 2, 'area': 120, 'num_garages': 1},
    {'provincia': 'Pichincha',   'lugar': 'Quito',      'num_dormitorios': 4, 'num_banos': 3, 'area': 200, 'num_garages': 2},
    {'provincia': 'Guayas',      'lugar': 'Guayaquil',  'num_dormitorios': 2, 'num_banos': 1, 'area': 70,  'num_garages': 0},
    {'provincia': 'El Oro',      'lugar': 'Machala',    'num_dormitorios': 3, 'num_banos': 2, 'area': 150, 'num_garages': 2},
    {'provincia': 'Imbabura',    'lugar': 'Ibarra',     'num_dormitorios': 2, 'num_banos': 1, 'area': 60,  'num_garages': 0},
    {'provincia': 'Santa Elena', 'lugar': 'Salinas',    'num_dormitorios': 3, 'num_banos': 2, 'area': 100, 'num_garages': 1},
])

escenarios['precio_predicho_USD'] = modelo_cargado.predict(escenarios[FEATURES]).round(2)
print('=== Predicciones para multiples escenarios ===')
print(escenarios.to_string(index=False))

## 9. Resumen

| Elemento | Detalle |
|---|---|
| **Modelo** | Gradient Boosting Regressor |
| **Transformacion** | log1p(precio) - expm1 en prediccion |
| **Pipeline** | OrdinalEncoder + StandardScaler + GBM |
| **Categorias desconocidas** | Manejadas con `unknown_value=-1` |
| **Archivo serializado** | `model/modelo_precio.pkl` |

### Por que este R2 es razonable para este dataset

El precio de alquiler tiene alta variabilidad inherente que **no puede ser capturada** por las variables disponibles:
- Piso del departamento, vista, estado de conservacion
- Amenidades (piscina, gimnasio, seguridad)
- Antiguedad del inmueble
- Negociacion individual entre arrendador y arrendatario

Con solo 6 features y 500 registros, un R2 ~ 0.50 en CV es un resultado **solido y honesto**.

> **Proximo paso:** Consumir el modelo desde la API REST -> `app.py`